# Preprocessing

### Load and Convert Fine-tuned Transformer Model to TransformerLens

In [ ]:
from src import load_finetuned_model

base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")

model.eval()

## Find the data with the correct answer

In [ ]:
from src import filter_correct_data
import pandas as pd

dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_corrected.csv"
filtered_data_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered_AOS.csv"
test_df = pd.read_csv(dataset_path)

df_filtered = filter_correct_data(model, test_df, 
                                  "original_sentence", "original_triplet", filter_mode="AOS", filter_only_correct=False, 
                                  save_path=filtered_data_path)

## Create EAP Dataset

#### Building the Dataset

In [ ]:
import src
import importlib
importlib.reload(src.utils)

from src.utils import build_eap_dataset
import pandas as pd


filtered_data = pd.read_csv(
    "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered_S.csv")

eap_df = build_eap_dataset(
    model=model,
    df=filtered_data,
    sentence_col="original_sentence",
    triplet_col="original_triplet",
    corrupted_col="counterfact3_modified",
    corrupted_triplet_col="counterfact_triplet3_modified",
    suffix="[S]",
    filer_same_length_counterfactuals=True
)
eap_df.to_csv("eap_dataset/eap_dataset_sentiment_multitokens.csv", index=False)

# EAP-IG

In [ ]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()

In [ ]:
import argparse
import ast
import os
from functools import partial
from random import random
from typing import Optional

import pandas as pd
import transformers
from torch.utils.data import Dataset, DataLoader
import torch
from typing_extensions import Tuple, List, Union

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline, evaluate_baseline_multitoken, evaluate_graph_multitoken
from eap.attribute import attribute
from src.utils import build_eap_dataset
from src import load_finetuned_model, filter_correct_data
from src.metric import logit_diff

In [ ]:
# load model
base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True
model.cfg.ungroup_grouped_query_attention = True

In [ ]:
# load dataset
ds = pd.read_csv("eap_dataset/eap_dataset_aspect_multitokens.csv")

In [ ]:
g = Graph.from_model(model)

In [ ]:
baseline = evaluate_baseline_multitoken(
    model,
    df=ds,
    metrics=[logit_diff],
    run_corrupted=False,
    batch_size=4
)
print(f"Original performance is logit_dif={baseline}")

In [ ]:
attribute(
    model=model,
    graph=g,
    dataloader=ds,  # can be Dataset or DataLoader
    metric=partial(logit_diff,loss=False, mean=True),
    method="EAP-IG-inputs",
    ig_steps=5,
    is_absa=True,
    batch_size=5,
    device="mps"
)

In [ ]:
n_edges = g.real_edge_mask.sum().item()  # total 171K edges for qwen2.5-0.5B
for topk in [100, 200, 500, 1000, 2000, 5000, 10000, 20000]:
    g.reset()
    g.apply_topn(topk, True)
    results = evaluate_graph_multitoken(model=model,
                                        graph=g,
                                        df=ds,  # your full dataset DataFrame
                                        metrics=[partial(logit_diff, mean=True, loss=False)],  # or just [logit_diff]
                                        batch_size=4,)

    print(f"with top-k = {topk} ({topk/n_edges:.1%}), the circuit's performance is {results}, faithfulness={results/baseline:.1%}")
    g.to_pt(f'outputs/opinion_circuit_topk-{topk}.pt')

    print(f"included nodes: {g.count_included_nodes()}, included edges: {g.count_included_edges()}")

In [ ]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()


!python run_eap_multitokens.py \
  --base_model "Qwen/Qwen2.5-0.5B" \
  --finetuned_model "models/fine_tuned_model/" \
  --dataset "eap_dataset/eap_dataset_aspect_multitokens.csv" \
  --output_dir "outputs/multitokens" \
  --batch_size 4 \
  --ig_steps 5 \
  --topks 100 200 500 1000 2000 5000 10000 20000 30000 40000 \
  --device "mps" \
  --element "aspect" \
  --log_file "multitokens_faithfulness_log.csv"

In [ ]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()

!python run_eap_multitokens.py \
  --base_model "Qwen/Qwen2.5-0.5B" \
  --finetuned_model "models/fine_tuned_model/" \
  --dataset "eap_dataset/eap_dataset_opinion_multitokens.csv" \
  --output_dir "outputs/multitokens" \
  --batch_size 4 \
  --ig_steps 5 \
  --topks 100 200 500 1000 2000 5000 10000 20000 30000 40000 \
  --device "mps" \
  --element "opinion" \
  --log_file "multitokens_faithfulness_log.csv"

In [ ]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()

!python run_eap_multitokens.py \
  --base_model "Qwen/Qwen2.5-0.5B" \
  --finetuned_model "models/fine_tuned_model/" \
  --dataset "eap_dataset/eap_dataset_sentiment_multitokens.csv" \
  --output_dir "outputs/multitokens" \
  --batch_size 4 \
  --ig_steps 5 \
  --topks 100 200 500 1000 2000 5000 10000 20000 30000 40000 \
  --device "mps" \
  --element "sentiment" \
  --log_file "multitokens_faithfulness_log.csv"

# Circuit Merging

In [ ]:
from src.utils import edge_merging

graph_paths = ["outputs/multitokens/aspect_circuit_topk-5000.pt", "outputs/multitokens/sentiment_circuit_topk-5000.pt", "outputs/multitokens/opinion_circuit_topk-5000.pt"]

complete_edges = edge_merging(graph_paths=graph_paths)

complete_edges.to_csv("outputs/multitokens/complete_circuit_topk-5000.csv", index=None)

print(complete_edges.shape)

In [2]:
import pandas as pd
from eap.graph import Graph

df = pd.read_csv("outputs/multitokens/complete_circuit_topk-2000.csv")
circuit_path = "outputs/multitokens/aspect_circuit_topk-2000.pt"
g = Graph.from_pt(circuit_path)
print(g.count_included_edges())
number_of_edge = g.count_included_edges()
for i, edge in enumerate(g.edges.values()):
    if edge.in_graph != True:
        if "m" in edge.child.name:
            mask = (df['parent_node'] == edge.parent.name) & (df['child_node'] == edge.child.name)
        else:
            mask = (df['parent_node'] == edge.parent.name) & (df['child_node'] == edge.child.name) & (df['child_type'] == edge.qkv)
        if sum(mask) == 1:
            edge.in_graph = True
            number_of_edge += 1
            print(i, number_of_edge, end="\r")
g.to_pt("outputs/multitokens/complete_circuit_topk-2000.pt")

1730


In [ ]:
topk = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 30000, 40000]

for t in topk:
    print(t)
    df = pd.read_csv(f"outputs/multitokens/complete_circuit_topk-{t}.csv")

    a = []
    for row in df.iterrows():
        if "m" not in row[1]["child_node"] and "logits" not in row[1]["child_node"]:
            a.append(row[1]['child_node'] + ' ' + row[1]['child_type'])

    print(len(np.unique(a)))
    print(df.shape)
    print((df.shape[0]/179387)*100)
    print("\n")

# SFT

## SFT with Active Nodes

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformer_lens import HookedTransformer
from transformer_lens.train import train
from transformer_lens.train import HookedTransformerTrainConfig
import pandas as pd
import gc
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

gc.collect()

torch.mps.empty_cache()


csv_path = "outputs/multitokens/complete_circuit_topk-2000.csv"
# json_path = "hotel_dataset/hotel_aste_formatted.json"
json_path = "hotel_dataset/hotel_aste_dev_augmented.json"
model_name = "Qwen/Qwen2.5-0.5B"

# === Load ABSA Dataset ===
with open(json_path) as f:
    absa_data = json.load(f)

# === Tokenizer setup ===
model = HookedTransformer.from_pretrained(model_name)  # or your fine-tuned base

# === Custom Dataset ===
class ABSADataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        input_enc = self.tokenizer(
            sample["input"], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        )
        target_enc = self.tokenizer(
            sample["target"], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": input_enc["input_ids"].squeeze(),
            "labels": target_enc["input_ids"].squeeze(),
        }

dataset = ABSADataset(absa_data[:500], model.tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# === Parse Active Edges ===
df = pd.read_csv(csv_path)
attention_targets = set()
mlp_layers = set()

for _, row in df.iterrows():
    if isinstance(row['child_node'], str) and row['child_node'].startswith('a'):
        layer = int(row['child_node'].split('.')[0][1:])
        head = int(row['child_node'].split('.')[1][1:])
        typ = row['child_type']
        attention_targets.add((layer, head, typ))
    elif isinstance(row['child_node'], str) and row['child_node'].startswith('m'):
        mlp_layers.add(int(row['child_node'][1:]))

# === Freeze all parameters ===
for name, param in model.named_parameters():
    param.requires_grad = False

# === Unfreeze only the targeted parameters ===
def unfreeze_param_subsection(param_tensor, head_index, head_dim):
    param_tensor.requires_grad = True
    param_tensor.data[head_index * head_dim : (head_index + 1) * head_dim].requires_grad = True

head_dim = model.cfg.d_head

for layer, head, typ in attention_targets:
    if typ == 'q':
        unfreeze_param_subsection(model.blocks[layer].attn.W_Q, head, head_dim)
    elif typ == 'k':
        unfreeze_param_subsection(model.blocks[layer].attn.W_K, head, head_dim)
    elif typ == 'v':
        unfreeze_param_subsection(model.blocks[layer].attn.W_V, head, head_dim)

for layer in mlp_layers:
    model.blocks[layer].mlp.W_in.requires_grad = True
    model.blocks[layer].mlp.W_out.requires_grad = True

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
model.train()

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=model.tokenizer.pad_token_id)

for epoch in range(3):
    print(f"Epoch {epoch + 1}")
    for step, batch in enumerate(tqdm(dataloader, desc=f"Training Epoch {epoch + 1}")):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, return_type="logits")
        logits = outputs  # shape: [batch_size, seq_len, vocab_size]
        loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 5 == 0:
            print(f"Epoch {epoch} Step {step} Loss: {loss.item():.4f}")


In [ ]:
model.generate([ "airnya kurang kencang . [A] [O] [S]"], 
               max_new_tokens=50,
               stop_at_eos=True,
               return_type="str")

## SFT With Ablation

In [ ]:
from src.utils import load_model

import gc, torch
gc.collect()

torch.mps.empty_cache()


model = load_model("Qwen/Qwen2.5-0.5B")
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True
model.cfg.ungroup_grouped_query_attention = True

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformer_lens import HookedTransformer
from transformer_lens.train import train
from transformer_lens.train import HookedTransformerTrainConfig
import pandas as pd
import gc
import torch.nn as nn
import torch.optim as optim
from eap.graph import Graph
from tqdm import tqdm

class ABSADataset_2(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        return {
            "input_ids": sample["input"],
            "labels": sample["target"],
        }

In [ ]:
graph = Graph.from_pt("outputs/multitokens/complete_circuit_topk-20000.pt")


csv_path = "outputs/multitokens/complete_circuit_topk-20000.csv"
json_path_intervention = "hotel_dataset/hotel_aste_formatted.json"
json_path = "hotel_dataset/hotel_aste_dev_augmented.json"
model_name = "Qwen/Qwen2.5-0.5B"

with open(json_path) as f:
    absa_data = json.load(f)

dataset = ABSADataset_2(absa_data[:100], model.tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

intervention_dataloader = None
# with open(json_path_intervention) as f:
#     intervention_data = json.load(f)

# intervention_dataset = ABSADataset_2(intervention_data, model.tokenizer)
# intervention_dataloader = DataLoader(intervention_dataset, batch_size=4, shuffle=True)

In [ ]:
from eap.attribute import tokenize_plus, make_hooks_and_matrices, compute_mean_activations
from einops import einsum
from eap.graph import Graph, AttentionNode
import torch.nn.functional as F
gc.collect()

torch.mps.empty_cache()

intervention = "zero"

assert model.cfg.use_attn_result, "Model must be configured to use attention result (model.cfg.use_attn_result)"
if model.cfg.n_key_value_heads is not None:
    assert model.cfg.ungroup_grouped_query_attention, "Model must be configured to ungroup grouped attention (model.cfg.ungroup_grouped_attention)"
    
assert intervention in ['patching', 'zero', 'mean', 'mean-positional'], f"Invalid intervention: {intervention}"

if 'mean' in intervention:
    assert intervention_dataloader is not None, "Intervention dataloader must be provided for mean interventions"
    per_position = 'positional' in intervention

    means = compute_mean_activations(model, graph, intervention_dataloader, per_position=per_position)
    means = means.unsqueeze(0)
    if not per_position:
        means = means.unsqueeze(0)

# This step cleans up the graph, removing components until it's fully connected
graph.prune()

# Construct a matrix that indicates which edges are in the graph
in_graph_matrix = graph.in_graph.to(device=model.cfg.device, dtype=model.cfg.dtype)

# same thing but for neurons
if graph.neurons_in_graph is not None:
    neuron_matrix = graph.neurons_in_graph.to(device=model.cfg.device, dtype=model.cfg.dtype)

    # If an edge is in the graph, but not all its neurons are, we need to update that edge anyway
    node_fully_in_graph = (neuron_matrix.sum(-1) == model.cfg.d_model).to(model.cfg.dtype)
    in_graph_matrix = einsum(in_graph_matrix, node_fully_in_graph, 'forward backward, forward -> forward backward')
else:
    neuron_matrix = None

# We take the opposite matrix, because we'll use it as a mask to specify 
# which edges we want to corrupt
in_graph_matrix = 1 - in_graph_matrix
if neuron_matrix is not None:
    neuron_matrix = 1 - neuron_matrix
    
if model.cfg.use_normalization_before_and_after:
    # If the model also normalizes the outputs of attention heads, we'll need to take that into account when evaluating the graph.
    attention_head_mask = torch.zeros((graph.n_forward, model.cfg.n_layers), device='cuda', dtype=model.cfg.dtype)
    for node in graph.nodes.values():
        if isinstance(node, AttentionNode):
            attention_head_mask[graph.forward_index(node), node.layer] = 1

    non_attention_head_mask = 1 - attention_head_mask.any(-1).to(dtype=model.cfg.dtype)
    attention_biases = torch.stack([block.attn.b_O for block in model.blocks])


def make_input_construction_hook(activation_matrix, in_graph_vector, neuron_matrix):
    def input_construction_hook(activations, hook):
        # Case where layernorm is applied after attention (gemma only)
        if model.cfg.use_normalization_before_and_after:
            activation_differences = activation_matrix[0] - activation_matrix[1]
            
            # get the clean outputs of the attention heads that came before
            clean_attention_results = einsum(activation_matrix[1, :, :, :len(in_graph_vector)], attention_head_mask[:len(in_graph_vector)], 'batch pos previous hidden, previous layer -> batch pos layer hidden')
            
            # get the update corresponding to non-attention heads, and the difference between clean and corrupted attention heads
            if neuron_matrix is not None:
                non_attention_update = einsum(activation_differences[:, :, :len(in_graph_vector)], neuron_matrix[:len(in_graph_vector)], in_graph_vector, non_attention_head_mask[:len(in_graph_vector)], 'batch pos previous hidden, previous hidden, previous ..., previous -> batch pos ... hidden')
                corrupted_attention_difference = einsum(activation_differences[:, :, :len(in_graph_vector)], neuron_matrix[:len(in_graph_vector)], in_graph_vector, attention_head_mask[:len(in_graph_vector)], 'batch pos previous hidden, previous hidden, previous ..., previous layer -> batch pos ... layer hidden')                    
            else:
                non_attention_update = einsum(activation_differences[:, :, :len(in_graph_vector)], in_graph_vector, non_attention_head_mask[:len(in_graph_vector)], 'batch pos previous hidden, previous ..., previous -> batch pos ... hidden')
                corrupted_attention_difference = einsum(activation_differences[:, :, :len(in_graph_vector)], in_graph_vector, attention_head_mask[:len(in_graph_vector)], 'batch pos previous hidden, previous ..., previous layer -> batch pos ... layer hidden')
            
            # add the biases to the attention results, and compute the corrupted attention results using the difference
            # we process all the attention heads at once; this is how we can tell if we're doing that
            if in_graph_vector.ndim == 2:
                corrupted_attention_results = clean_attention_results.unsqueeze(2) + corrupted_attention_difference
                # (1, 1, 1, layer, hidden)
                clean_attention_results += attention_biases.unsqueeze(0).unsqueeze(0)
                corrupted_attention_results += attention_biases.unsqueeze(0).unsqueeze(0).unsqueeze(0)
            else:
                corrupted_attention_results = clean_attention_results + corrupted_attention_difference
                clean_attention_results += attention_biases.unsqueeze(0).unsqueeze(0)
                corrupted_attention_results += attention_biases.unsqueeze(0).unsqueeze(0)
            
            # pass both the clean and corrupted attention results through the layernorm and 
            # add the difference to the update
            update = non_attention_update
            valid_layers = attention_head_mask[:len(in_graph_vector)].any(0)
            for i, valid_layer in enumerate(valid_layers):
                if not valid_layer:
                    break
                if in_graph_vector.ndim == 2:
                    update -= model.blocks[i].ln1_post(clean_attention_results[:, :, None, i])
                    update += model.blocks[i].ln1_post(corrupted_attention_results[:, :, :, i])                        
                else:
                    update -= model.blocks[i].ln1_post(clean_attention_results[:, :, i])
                    update += model.blocks[i].ln1_post(corrupted_attention_results[:, :, i])
                    
        else:
            # In the non-gemma case, things are easy!
            activation_differences = activation_matrix
            # The ... here is to account for a potential head dimension, when constructing a whole attention layer's input
            if neuron_matrix is not None:
                update = einsum(activation_differences[:, :, :len(in_graph_vector)], neuron_matrix[:len(in_graph_vector)], in_graph_vector,'batch pos previous hidden, previous hidden, previous ... -> batch pos ... hidden')
            else:
                update = einsum(activation_differences[:, :, :len(in_graph_vector)], in_graph_vector,'batch pos previous hidden, previous ... -> batch pos ... hidden')
        activations += update
        return activations
    return input_construction_hook


def make_input_construction_hooks(activation_differences, in_graph_matrix, neuron_matrix):
    input_construction_hooks = []
    for layer in range(model.cfg.n_layers):
        # If any attention node in the layer is in the graph, just construct the input for the entire layer
        if any(graph.nodes[f'a{layer}.h{head}'].in_graph for head in range(model.cfg.n_heads)) and not (neuron_matrix is None and all(parent_edge.in_graph for head in range(model.cfg.n_heads) for parent_edge in graph.nodes[f'a{layer}.h{head}'].parent_edges)):
            for i, letter in enumerate('qkv'):
                node = graph.nodes[f'a{layer}.h0']
                prev_index = graph.prev_index(node)
                bwd_index = graph.backward_index(node, qkv=letter, attn_slice=True)
                input_cons_hook = make_input_construction_hook(activation_differences, in_graph_matrix[:prev_index, bwd_index], neuron_matrix)
                input_construction_hooks.append((node.qkv_inputs[i], input_cons_hook))
                
        # add MLP hook if MLP in graph
        if graph.nodes[f'm{layer}'].in_graph and not (neuron_matrix is None and all(parent_edge.in_graph for parent_edge in graph.nodes[f'm{layer}'].parent_edges)):
            node = graph.nodes[f'm{layer}']
            prev_index = graph.prev_index(node)
            bwd_index = graph.backward_index(node)
            input_cons_hook = make_input_construction_hook(activation_differences, in_graph_matrix[:prev_index, bwd_index], neuron_matrix)
            input_construction_hooks.append((node.in_hook, input_cons_hook))
                
    # Always add the logits hook
    if not (neuron_matrix is None and all(parent_edge.in_graph for parent_edge in graph.nodes['logits'].parent_edges)):
        node = graph.nodes['logits']
        fwd_index = graph.prev_index(node)
        bwd_index = graph.backward_index(node)
        input_cons_hook = make_input_construction_hook(activation_differences, in_graph_matrix[:fwd_index, bwd_index], neuron_matrix)
        input_construction_hooks.append((node.in_hook, input_cons_hook))

    return input_construction_hooks

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=model.tokenizer.pad_token_id)
model.train()

n_epochs = 3
for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1}")
    for step, batch in enumerate(tqdm(dataloader, desc=f"Training Epoch {epoch + 1}")):
        input_ids = batch["input_ids"]
        labels = batch["labels"]

        clean_tokens, attention_mask, input_lengths, n_pos = tokenize_plus(model, input_ids, max_length=128)
        label_tokens, _, _, _  = tokenize_plus(model, labels, max_length=128)

        max_len = max(clean_tokens.size(1), label_tokens.size(1))

        # Pad input
        if clean_tokens.size(1) < max_len:
            pad_len = max_len - clean_tokens.size(1)
            clean_tokens = F.pad(clean_tokens, (0, pad_len), value=model.tokenizer.pad_token_id)
            attention_mask = F.pad(attention_mask, (0, pad_len), value=0)

        # Pad labels with -100 so they're ignored in loss
        if label_tokens.size(1) < max_len:
            pad_len = max_len - label_tokens.size(1)
            label_tokens = F.pad(label_tokens, (0, pad_len), value=-100)

        seq_len = clean_tokens.shape[1]
        (fwd_hooks_corrupted, fwd_hooks_clean, _), activation_difference = make_hooks_and_matrices(model, graph, len(input_ids), seq_len, None)

        input_construction_hooks = make_input_construction_hooks(activation_difference, in_graph_matrix, neuron_matrix)

        with model.hooks(fwd_hooks_clean + input_construction_hooks):
            logits = model(clean_tokens, attention_mask=attention_mask)
            loss = loss_fn(logits.view(-1, logits.size(-1)), label_tokens.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if step % 5 == 0:
            print(f"Epoch {epoch} Step {step} Loss: {loss.item():.4f}")
